# Pemeriksaan data kerusakan PART - OMEXP
Laporan ini memeriksa kondisi PART setiap 30 hari. Satu pemeriksaan 30-harian disebut **snapshot**, yaitu catatan keadaan PART pada satu tanggal. Pertanyaan utamanya: apakah PART akan mengalami kerusakan dalam 30 hari setelah tanggal tersebut? Kerusakan dihitung ketika terdapat `DISMANTLED + CORRECTIVE`, atau ketika `DISMANTLED + PREVENTIVE` kemudian dikonfirmasi `BROKEN` atau `UNREPAIRABLE` sebelum pemasangan berikutnya. Kegiatan administrasi `RECON` tidak dihitung sebagai waktu kerja atau kerusakan. Laporan ini hanya membaca data dan tidak mengubah tabel sumber.

In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
PROJECT_DIR = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR / 'src'))
from database import connect
sns.set_theme(style='whitegrid')
def query(sql, params=None):
    with connect() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params or ())
            return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

## Panduan awal: urutan membaca EDA (silabus singkat)
Laporan ini sebaiknya dibaca berurutan sebagai berikut:

1. **Kenali isi database**: berapa model, unit PART, TERMINAL, lokasi, klien, event, cycle, dan failure.
2. **Pahami definisi**: bedakan model dengan unit fisik, event dengan cycle, serta failure dengan status lanjutan.
3. **Periksa kualitas data**: cari tanggal tidak valid, model/lokasi yang tidak cocok master, duplikasi, dan histori yang terpotong.
4. **Bentuk riwayat operasional**: keluarkan RECON dari perhitungan waktu, lalu susun event sesuai urutan waktu.
5. **Bentuk cycle pemasangan**: satu cycle dimulai ketika PART dipasang dan berakhir ketika failure, dipasang ulang, atau data berakhir.
6. **Bentuk snapshot dan target**: kondisi PART dicatat setiap 30 hari, lalu diperiksa apakah failure terjadi dalam 30 hari berikutnya.
7. **Bandingkan model dan lokasi**: hitung risiko dengan penyebut yang adil dan jumlah sampel minimum.
8. **Periksa fitur dan kestabilan waktu**: pastikan fitur hanya memakai masa lalu dan pola 2013-2024 tidak langsung dianggap sama dengan 2025-2026.
9. **Ambil keputusan**: tentukan data yang siap untuk baseline model dan data yang masih membutuhkan review manual.

## Glosarium istilah yang digunakan

| Istilah | Arti sederhana |
|---|---|
| **Model item** | Jenis atau tipe barang. Satu model dapat dimiliki banyak unit fisik. |
| **Unit item** | Satu barang fisik yang dibedakan dengan identifier atau serial. |
| **PART** | Komponen yang dipasang pada perangkat atau lokasi dan menjadi objek utama prediksi failure. |
| **TERMINAL** | Perangkat terminal. Jumlah dan pola failure-nya dipisahkan dari PART. |
| **Master data** | Daftar resmi model, lokasi, klien, status, dan jenis pekerjaan yang menjadi acuan validasi. |
| **Journey/event** | Satu catatan kegiatan atau perubahan status sebuah item pada suatu waktu. |
| **Operational event** | Event dengan waktu yang layak dipakai. RECON administratif dan tanggal tidak valid tidak ikut menghitung durasi. |
| **Installation cycle** | Periode sejak PART dipasang sampai failure pertama, pemasangan berikutnya, atau akhir data. |
| **Failure onset** | Waktu awal PART dianggap berhenti beroperasi karena kerusakan. |
| **Failure outcome** | Status yang mengonfirmasi hasil kerusakan, misalnya BROKEN atau UNREPAIRABLE. |
| **Snapshot** | Foto kondisi data PART pada satu tanggal, bukan foto gambar. Satu PART dapat memiliki banyak snapshot. |
| **Target 30 hari** | Pertanyaan apakah failure terjadi setelah snapshot dan paling lambat 30 hari berikutnya. |
| **Positif** | Snapshot yang benar-benar diikuti failure dalam 30 hari. |
| **Negatif** | Snapshot yang tidak diikuti failure dan memiliki bukti follow-up penuh selama 30 hari. |
| **Positive rate** | Jumlah positif dibagi seluruh observasi yang layak. Nilainya berubah menurut unit hitung: snapshot, cycle, atau unit PART. |
| **Class imbalance** | Jumlah label positif dan negatif sangat tidak seimbang. Akurasi biasa dapat terlihat bagus walaupun model gagal menemukan failure. |
| **Korelasi** | Ukuran apakah dua fitur cenderung berubah bersama. Korelasi tinggi dapat menunjukkan informasi yang berulang. |
| **Multikolinearitas/redundansi** | Beberapa fitur membawa informasi yang hampir sama, misalnya hitungan 90 hari dan 180 hari. Tidak selalu salah, tetapi perlu diseleksi atau diregularisasi. |
| **Information Value (IV)** | Screening satu fitur terhadap target. Nilai lebih besar berarti pemisahan awal lebih kuat, tetapi bukan bukti kausal atau feature importance model final. |
| **Drift / PSI** | Perubahan distribusi fitur terhadap periode referensi. PSI di bawah 0,10 relatif stabil; 0,10-0,25 perlu dipantau; minimal 0,25 menunjukkan drift besar. |
| **Missing struktural** | Nilai kosong yang mempunyai arti, misalnya hari sejak failure terakhir kosong karena PART memang belum pernah failure. |
| **Follow-up tidak lengkap** | Belum tersedia cukup data setelah snapshot atau failure untuk memastikan kejadian selanjutnya. |
| **Right-censored** | Cycle masih berjalan ketika database berhenti; akhir umur PART belum diketahui. |
| **Cohort valid** | Kelompok cycle PART yang lolos aturan awal: identifier, model, dan waktu pemasangan dapat dipercaya. |
| **Fitur** | Informasi yang tersedia pada tanggal snapshot dan dapat menjadi masukan model, misalnya umur PART, model, lokasi, atau corrective sebelumnya. |
| **Lokasi canonical** | Nama lokasi yang sudah dicocokkan dengan master lokasi resmi melalui exact match, alias kontekstual terverifikasi, atau fuzzy berkeyakinan tinggi. |
| **Fuzzy score** | Nilai kemiripan teks 0-100%. Auto-mapping memerlukan skor minimal 90% dan selisih minimal 8 poin dari kandidat kedua. |
| **Fuzzy review** | Kandidat yang belum aman dipetakan otomatis karena skor rendah atau dua kandidat terlalu berdekatan. |
| **Data leakage** | Kesalahan ketika informasi masa depan ikut dipakai untuk memprediksi masa depan. |

In [ ]:
database_overview = query("""
WITH label_gap AS (
    SELECT c.installation_cycle_id,
        BOOL_OR(o.event_semantic = 'RETURN_FLOW') has_return,
        BOOL_OR(o.event_semantic = 'FAILURE_OUTCOME') has_failure_outcome
    FROM analytics.item_installation_cycle c
    JOIN analytics.item_journey_operational_timeline o
      ON o.item_identifier_clean = c.item_identifier_clean
     AND o.created_on > c.installed_on AND o.created_on <= c.cycle_end_on
    WHERE NOT c.has_observed_failure
      AND o.event_semantic IN ('RETURN_FLOW', 'FAILURE_OUTCOME')
    GROUP BY c.installation_cycle_id
)
SELECT * FROM (
 SELECT 1 urutan, 'Item yang muncul di journal' kelompok, 'Model item yang digunakan' ukuran, (SELECT COUNT(DISTINCT item_model_code_clean) FROM analytics.item_journey_clean)::bigint jumlah, 'Distinct seluruh model pada journey; bukan jumlah unit fisik' keterangan
 UNION ALL SELECT 2, 'Item yang muncul di journal', 'Model PART yang digunakan', (SELECT COUNT(DISTINCT item_model_code_clean) FROM analytics.item_journey_clean WHERE item_category_clean='PART'), 'Model yang benar-benar muncul pada event PART'
 UNION ALL SELECT 3, 'Item yang muncul di journal', 'Model TERMINAL yang digunakan', (SELECT COUNT(DISTINCT item_model_code_clean) FROM analytics.item_journey_clean WHERE item_category_clean='TERMINAL'), 'Model yang benar-benar muncul pada event TERMINAL'
 UNION ALL SELECT 4, 'Item yang muncul di journal', 'Identifier PART yang digunakan', (SELECT COUNT(DISTINCT item_identifier_clean) FROM analytics.item_journey_clean WHERE item_category_clean='PART' AND item_identifier_clean IS NOT NULL), 'Unit PART yang mempunyai sedikitnya satu event'
 UNION ALL SELECT 5, 'Item yang muncul di journal', 'Identifier TERMINAL yang digunakan', (SELECT COUNT(DISTINCT item_identifier_clean) FROM analytics.item_journey_clean WHERE item_category_clean='TERMINAL' AND item_identifier_clean IS NOT NULL), 'Unit TERMINAL yang mempunyai sedikitnya satu event'
 UNION ALL SELECT 6, 'Item yang muncul di journal', 'Lokasi valid yang digunakan', (SELECT COUNT(DISTINCT place_canonical_clean) FROM analytics.item_journey_clean WHERE place_canonical_clean IS NOT NULL), 'Lokasi journey yang berhasil dicocokkan ke master'
 UNION ALL SELECT 7, 'Item yang muncul di journal', 'Klien valid yang digunakan', (SELECT COUNT(DISTINCT client_canonical_clean) FROM analytics.item_journey_clean WHERE client_canonical_clean IS NOT NULL), 'Klien canonical setelah exact/fuzzy mapping aman'
 UNION ALL SELECT 8, 'Aktivitas', 'Seluruh journey/event mentah', (SELECT COUNT(*) FROM analytics.item_journey_clean), 'Termasuk RECON dan event dengan tanggal bermasalah'
 UNION ALL SELECT 9, 'Aktivitas', 'Event operasional', (SELECT COUNT(*) FROM analytics.item_journey_operational_timeline), 'Event yang layak dipakai menyusun urutan waktu'
 UNION ALL SELECT 10, 'Aktivitas', 'Work order', (SELECT COUNT(*) FROM analytics.work_order_clean), 'Dokumen pekerjaan yang tersedia'
 UNION ALL SELECT 11, 'Aktivitas', 'Riwayat status work order', (SELECT COUNT(*) FROM analytics.work_order_history_clean), 'Perubahan status dari seluruh work order'
 UNION ALL SELECT 12, 'Cycle dan failure', 'Event pemasangan', (SELECT COUNT(*) FROM analytics.item_journey_operational_timeline WHERE status_clean='INSTALLED'), 'Satu PART dapat dipasang lebih dari sekali'
 UNION ALL SELECT 13, 'Cycle dan failure', 'Seluruh installation cycle', (SELECT COUNT(*) FROM analytics.item_installation_cycle), 'Cycle sejak pemasangan sampai failure, reinstall, atau cutoff'
 UNION ALL SELECT 14, 'Cycle dan failure', 'Cycle cohort valid', (SELECT COUNT(*) FROM analytics.item_installation_cycle WHERE is_initial_model_cohort), 'Cycle PART yang lolos pemeriksaan awal'
 UNION ALL SELECT 15, 'Cycle dan failure', 'Seluruh failure', (SELECT COUNT(*) FROM analytics.failure_event_clean), 'Corrective dismantle ditambah preventive yang dikonfirmasi rusak'
 UNION ALL SELECT 16, 'Cycle dan failure', 'Failure pada PART', (SELECT COUNT(*) FROM analytics.failure_event_clean WHERE item_category_clean='PART'), 'Failure dengan kategori PART'
 UNION ALL SELECT 17, 'Cycle dan failure', 'Failure pada TERMINAL', (SELECT COUNT(*) FROM analytics.failure_event_clean WHERE item_category_clean='TERMINAL'), 'Dipisahkan dari model failure PART'
 UNION ALL SELECT 18, 'Cycle dan failure', 'Cycle valid yang berakhir failure', (SELECT COUNT(*) FROM analytics.item_installation_cycle WHERE is_initial_model_cohort AND has_observed_failure), 'Failure yang masuk cohort snapshot utama'
 UNION ALL SELECT 19, 'Cycle dan failure', 'Failure dengan alur lanjutan terkonfirmasi', (SELECT COUNT(*) FROM analytics.failure_event_flow WHERE flow_confirmation_status <> 'OPEN_OR_INCOMPLETE_FLOW'), 'Ada RETURN atau proses repair yang cukup sebagai konfirmasi'
 UNION ALL SELECT 20, 'Cycle dan failure', 'Failure tanpa alur lanjutan lengkap', (SELECT COUNT(*) FROM analytics.failure_event_flow WHERE flow_confirmation_status='OPEN_OR_INCOMPLETE_FLOW'), 'Tidak otomatis salah dan tidak menghapus label failure'
 UNION ALL SELECT 21, 'Cycle dan failure', 'Kemungkinan masih berjalan (0-30 hari)', (SELECT COALESCE(SUM(failure_count),0) FROM analytics.eda_incomplete_failure_summary WHERE followup_review_group='LIKELY_ONGOING_0_30D'), 'Terjadi dekat cutoff data'
 UNION ALL SELECT 22, 'Cycle dan failure', 'Perlu dipantau (31-180 hari)', (SELECT COALESCE(SUM(failure_count),0) FROM analytics.eda_incomplete_failure_summary WHERE followup_review_group='REVIEW_31_180D'), 'Belum cukup baru, tetapi belum pasti histori hilang'
 UNION ALL SELECT 23, 'Cycle dan failure', 'Kemungkinan histori hilang (>180 hari)', (SELECT COALESCE(SUM(failure_count),0) FROM analytics.eda_incomplete_failure_summary WHERE followup_review_group='LIKELY_HISTORY_GAP_GT_180D'), 'Prioritas untuk sampling manual'
 UNION ALL SELECT 24, 'Review label', 'Cycle RETURNED tanpa onset failure', (SELECT COUNT(*) FROM label_gap WHERE has_return), 'RETURNED saja belum membuktikan rusak'
 UNION ALL SELECT 25, 'Review label', 'Status rusak jelas tanpa onset tepercaya', (SELECT COUNT(*) FROM label_gap WHERE has_failure_outcome), 'Kandidat review, belum menjadi label utama'
 UNION ALL SELECT 26, 'Dataset model', 'Seluruh snapshot 30 hari', (SELECT COUNT(*) FROM analytics.item_observation_30d), 'Satu PART dapat muncul berkali-kali'
 UNION ALL SELECT 27, 'Dataset model', 'Snapshot layak training', (SELECT COUNT(*) FROM analytics.item_observation_30d WHERE is_training_eligible), 'Memiliki label positif atau follow-up negatif penuh'
 UNION ALL SELECT 28, 'Dataset model', 'Snapshot positif', (SELECT COUNT(*) FROM analytics.item_observation_30d WHERE is_training_eligible AND target_failure_30d), 'Failure terjadi dalam 30 hari berikutnya'
 UNION ALL SELECT 29, 'Dataset model', 'Snapshot negatif', (SELECT COUNT(*) FROM analytics.item_observation_30d WHERE is_training_eligible AND NOT target_failure_30d), 'Tidak failure dan follow-up 30 hari tersedia'
 UNION ALL SELECT 30, 'Dataset model', 'Snapshot dengan follow-up belum lengkap', (SELECT COUNT(*) FROM analytics.item_observation_30d WHERE NOT is_training_eligible), 'Dikeluarkan dari training'
 UNION ALL SELECT 31, 'Fitur lokasi', 'Snapshot dengan lokasi master valid', (SELECT COUNT(*) FROM analytics.item_observation_30d WHERE is_training_eligible AND is_location_feature_eligible), 'Boleh masuk analisis lokasi'
 UNION ALL SELECT 32, 'Fitur lokasi', 'Snapshot tanpa lokasi master', (SELECT COUNT(*) FROM analytics.item_observation_30d WHERE is_training_eligible AND NOT is_location_feature_eligible), 'Tetap diaudit, tetapi lokasi tidak dipakai sebagai fitur'
) ringkasan ORDER BY urutan
""")
database_overview['Jumlah'] = pd.to_numeric(database_overview['jumlah']).map(lambda x: f'{int(x):,}'.replace(',', '.'))
display(database_overview[['kelompok', 'ukuran', 'Jumlah', 'keterangan']].rename(columns={'kelompok': 'Kelompok', 'ukuran': 'Isi database', 'keterangan': 'Cara membaca'}))
modeling_period = query("SELECT MIN(observation_on)::date awal, MAX(observation_on)::date cutoff FROM analytics.item_observation_30d")
display(Markdown(f"**Periode dataset model:** {modeling_period.awal.iloc[0]:%d %B %Y} sampai **cutoff {modeling_period.cutoff.iloc[0]:%d %B %Y}**. Event setelah cutoff atau tanggal yang tidak dipercaya tidak ikut menghitung umur operasional."))

## 1. Apakah data sudah cukup aman untuk dianalisis?
Bagian ini menghitung jumlah siklus dan snapshot yang tersedia. Pemeriksaan tambahan memastikan tidak ada data ganda, tanggal yang berjalan mundur, atau jawaban masa depan yang tidak sengaja masuk sebagai informasi awal.

In [ ]:
readiness = query('SELECT * FROM analytics.eda_failure_readiness_summary ORDER BY metric')
metric_labels = {'all_observations': 'Semua snapshot', 'cycles_with_failure': 'Siklus yang mengalami kerusakan', 'excluded_incomplete_followup': 'Snapshot tanpa data 30 hari yang lengkap', 'installation_cycles': 'Semua siklus pemasangan', 'invalid_zero_duration_cycles': 'Siklus berdurasi nol (tidak valid)', 'positive_30d_observations': 'Snapshot diikuti kerusakan dalam 30 hari', 'right_censored_cycles': 'Siklus masih berjalan saat data berakhir', 'training_eligible_observations': 'Snapshot yang dapat dipakai untuk model', 'valid_model_cohort_cycles': 'Siklus PART yang lolos pemeriksaan awal'}
readiness_display = readiness.assign(metric=readiness['metric'].replace(metric_labels)).rename(columns={'metric': 'Ukuran yang diperiksa', 'value': 'Jumlah'})
display(readiness_display)
checks = query("""
SELECT
 COUNT(*) - COUNT(DISTINCT (installation_cycle_id, observation_on)) AS duplicate_keys,
 COUNT(*) FILTER (WHERE target_failure_30d AND NOT (next_failure_on > observation_on AND next_failure_on <= observation_on + INTERVAL '30 days')) AS invalid_positive_labels,
 COUNT(*) FILTER (WHERE days_since_installation < 0 OR days_since_last_event < 0 OR days_since_last_failure < 0) AS negative_time_features
FROM analytics.item_observation_30d
""")
display(checks.rename(columns={'duplicate_keys': 'Snapshot ganda', 'invalid_positive_labels': 'Label kerusakan tidak sesuai', 'negative_time_features': 'Perhitungan waktu negatif'}))
assert checks.iloc[0].eq(0).all(), 'Dataset gagal pemeriksaan leakage/key.'

## 2. Apakah target failure seimbang dengan non-failure?
Tabel pertama menunjukkan perbandingan label 1 dan 0 pada seluruh snapshot training. Karena failure adalah kejadian langka, akurasi biasa dapat menyesatkan: model yang selalu menjawab 'tidak rusak' bisa terlihat tinggi. Tabel dan grafik tahunan berikutnya memeriksa apakah positive rate juga berubah menurut waktu. Garis merah menandai awal pencatatan repair yang lebih rinci pada 2025.

In [ ]:
target_distribution = query('SELECT * FROM analytics.eda_target_class_distribution ORDER BY label_value')
target_display = target_distribution.rename(columns={'label_value': 'Label', 'label_name': 'Arti label', 'snapshot_count': 'Jumlah snapshot', 'class_percentage': 'Persentase (%)', 'negative_to_positive_ratio': 'Rasio negatif : positif', 'imbalance_status': 'Status imbalance'})
display(target_display)
positive_row = target_distribution.loc[target_distribution.label_value.eq(1)].iloc[0]
display(Markdown(f"**Kesimpulan imbalance:** hanya **{float(positive_row.class_percentage):.4f}%** snapshot yang positif. Terdapat sekitar **{float(positive_row.negative_to_positive_ratio):.2f} snapshot negatif untuk setiap 1 snapshot positif**. Karena itu evaluasi model nanti wajib memakai precision, recall, PR-AUC, ROC-AUC, calibration, dan confusion matrix; akurasi tidak boleh dipakai sendirian."))
plt.figure(figsize=(7, 4)); ax=sns.barplot(data=target_distribution, x='label_name', y='snapshot_count', hue='label_name', legend=False); ax.set_yscale('log'); ax.set(title='Jumlah snapshot per label (skala log)', xlabel='', ylabel='Jumlah snapshot'); plt.xticks(rotation=10); plt.tight_layout(); plt.show()
yearly = query('SELECT * FROM analytics.eda_failure_rate_by_year ORDER BY observation_year')
display(yearly.rename(columns={'observation_year': 'Tahun', 'observation_count': 'Jumlah snapshot', 'positive_count': 'Rusak dalam 30 hari', 'positive_percentage': 'Persentase positif'}))
ax = sns.lineplot(data=yearly, x='observation_year', y='positive_percentage', marker='o')
ax.axvline(2025, color='crimson', linestyle='--', label='Detailed repair era')
ax.set(title='Persentase snapshot yang diikuti kerusakan dalam 30 hari', ylabel='Persentase positif (%)', xlabel='Tahun snapshot')
ax.legend(); plt.show()

## 3. Apakah master lokasi/client dan informasi fitur cukup lengkap?
Cakupan master dihitung terhadap seluruh snapshot yang layak training, bukan hanya terhadap event mentah. `Unmatched` berarti nama canonical tidak tersedia. Kategori langka berarti kategori memiliki kurang dari 100 snapshot dan perlu digabung/diatur agar representasinya tidak terlalu sparse. Missing yang berarti 'belum pernah failure/corrective' diperlakukan berbeda dari data yang benar-benar hilang.

In [ ]:
master_coverage = query('SELECT * FROM analytics.eda_snapshot_master_coverage ORDER BY feature_name')
for col in ['total_snapshot','matched_snapshot','unmatched_snapshot','category_count','rare_category_count','rare_snapshot_count','unmatched_percentage','rare_snapshot_percentage']: master_coverage[col] = pd.to_numeric(master_coverage[col], errors='coerce')
coverage_display = master_coverage.rename(columns={'feature_name': 'Fitur master', 'total_snapshot': 'Snapshot training', 'matched_snapshot': 'Berhasil dipetakan', 'unmatched_snapshot': 'Belum dipetakan', 'category_count': 'Jumlah kategori', 'rare_category_count': 'Kategori langka (<100 snapshot)', 'rare_snapshot_count': 'Snapshot pada kategori langka', 'unmatched_percentage': 'Unmatched (%)', 'rare_snapshot_percentage': 'Kategori langka (%)', 'feature_decision': 'Keputusan'})
display(coverage_display)
coverage_plot = master_coverage.melt(id_vars='feature_name', value_vars=['unmatched_percentage','rare_snapshot_percentage'], var_name='coverage_issue', value_name='percentage')
coverage_plot['coverage_issue'] = coverage_plot.coverage_issue.map({'unmatched_percentage':'Belum cocok master','rare_snapshot_percentage':'Kategori langka'})
plt.figure(figsize=(7, 4)); sns.barplot(data=coverage_plot, x='feature_name', y='percentage', hue='coverage_issue').set(title='Dampak masalah master terhadap snapshot training', xlabel='Fitur', ylabel='Persentase snapshot (%)'); plt.tight_layout(); plt.show()
location_coverage = master_coverage.loc[master_coverage.feature_name.eq('LOCATION')].iloc[0]
display(Markdown(f"Lokasi belum cocok master hanya memengaruhi **{int(location_coverage.unmatched_snapshot):,} dari {int(location_coverage.total_snapshot):,} snapshot ({float(location_coverage.unmatched_percentage):.4f}%)**. Secara coverage, lokasi aman diuji sebagai prediktor, tetapi tetap gunakan kategori `UNKNOWN`, missing flag, minimum support, dan bandingkan model dengan-versus-tanpa lokasi.".replace(',', '.')))
missing = query('SELECT * FROM analytics.eda_feature_missingness ORDER BY missing_percentage DESC')
feature_labels = {'item_model_code_clean': 'Model PART', 'installed_client_clean': 'Client pemasangan', 'last_place_clean': 'Lokasi terakhir', 'days_since_installation': 'Hari sejak pemasangan', 'days_since_last_event': 'Hari sejak kegiatan terakhir', 'days_since_last_failure': 'Hari sejak kerusakan terakhir', 'days_since_last_corrective': 'Hari sejak corrective terakhir', 'days_at_last_location': 'Lama di lokasi terakhir'}
handling_labels = {'EXCLUDE_ROW_IF_MISSING_CORE_IDENTITY':'Keluarkan bila identitas inti kosong', 'UNKNOWN_CATEGORY_PLUS_MISSING_FLAG':'Gunakan kategori UNKNOWN dan penanda kosong', 'UNKNOWN_CATEGORY_PLUS_FLAG_COMPARE_WITHOUT_LOCATION':'UNKNOWN + penanda; bandingkan model tanpa lokasi', 'EXCLUDE_IF_MISSING_CYCLE_START':'Keluarkan bila awal cycle tidak diketahui', 'MEDIAN_BY_MODEL_PLUS_MISSING_FLAG':'Median per model + penanda kosong', 'STRUCTURAL_NO_PRIOR_FAILURE_USE_INDICATOR_AND_SENTINEL':'Bukan error: belum pernah failure; gunakan indikator + sentinel', 'STRUCTURAL_NO_PRIOR_CORRECTIVE_USE_INDICATOR_AND_SENTINEL':'Bukan error: belum pernah corrective; gunakan indikator + sentinel', 'MISSING_FLAG_AND_COMPARE_MODEL_WITHOUT_LOCATION_AGE':'Penanda kosong; bandingkan tanpa umur lokasi'}
missing['missing_percentage'] = pd.to_numeric(missing.missing_percentage, errors='coerce')
missing_display = missing.assign(feature_name=missing['feature_name'].replace(feature_labels), recommended_handling=missing['recommended_handling'].replace(handling_labels)).rename(columns={'feature_name': 'Informasi', 'missing_count': 'Jumlah kosong', 'missing_percentage': 'Persentase kosong', 'available_count': 'Jumlah tersedia', 'recommended_handling': 'Strategi sebelum modeling'})
display(missing_display[['Informasi','Jumlah kosong','Persentase kosong','Jumlah tersedia','Strategi sebelum modeling']])
plt.figure(figsize=(8, 5)); sns.barplot(data=missing_display, y='Informasi', x='Persentase kosong', color='steelblue').set(title='Persentase informasi yang masih kosong', xlabel='Kosong (%)', ylabel=''); plt.tight_layout(); plt.show()

## 4. Berapa lama PART bertahan sejak dipasang sampai rusak?
Hanya siklus dengan tanggal pemasangan yang dapat dipercaya yang digunakan. Median adalah nilai tengah dan biasanya lebih mudah dibaca daripada rata-rata ketika sebagian PART bertahan sangat lama.

In [ ]:
cycle_stats = query("""SELECT COUNT(*) failure_cycles, ROUND(AVG(days_installed_to_failure)::numeric,2) mean_days, ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY days_installed_to_failure)::numeric,2) median_days, ROUND(PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY days_installed_to_failure)::numeric,2) p90_days FROM analytics.item_installation_cycle WHERE is_initial_model_cohort AND has_observed_failure""")
display(cycle_stats.rename(columns={'failure_cycles': 'Siklus dengan kerusakan', 'mean_days': 'Rata-rata hari', 'median_days': 'Median hari', 'p90_days': '90% rusak sebelum hari ke-'}))
durations = query("SELECT LEAST(FLOOR(days_installed_to_failure / 90) * 90, 1800)::int bucket_days, COUNT(*) cycle_count FROM analytics.item_installation_cycle WHERE is_initial_model_cohort AND has_observed_failure GROUP BY 1 ORDER BY 1")
sns.barplot(data=durations, x='bucket_days', y='cycle_count', color='steelblue').set(title='Sebaran umur PART saat mengalami kerusakan', xlabel='Umur sejak dipasang (kelompok 90 hari)', ylabel='Jumlah siklus'); plt.xticks(rotation=45); plt.show()

## 5. Model PART mana yang lebih sering diikuti kerusakan?
Perbandingan hanya menampilkan model yang mempunyai data cukup: sedikitnya 20 PART dan 10 snapshot positif. Persentase tinggi belum otomatis berarti model tersebut penyebab kerusakan; umur, lokasi, dan riwayat pemakaian juga dapat berpengaruh.

In [ ]:
by_model = query("""SELECT item_model_code_clean, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items, COUNT(*) FILTER (WHERE target_failure_30d) positives, ROUND(100.0 * COUNT(*) FILTER (WHERE target_failure_30d) / COUNT(*), 3) positive_pct FROM analytics.item_observation_30d WHERE is_training_eligible GROUP BY 1 HAVING COUNT(DISTINCT item_identifier_clean) >= 20 AND COUNT(*) FILTER (WHERE target_failure_30d) >= 10 ORDER BY positives DESC LIMIT 25""")
display(by_model.rename(columns={'item_model_code_clean': 'Model PART', 'observations': 'Jumlah snapshot', 'items': 'Jumlah PART', 'positives': 'Rusak dalam 30 hari', 'positive_pct': 'Persentase positif'}))
sns.barplot(data=by_model, y='item_model_code_clean', x='positive_pct').set(title='Persentase kerusakan 30 hari menurut model PART', xlabel='Persentase positif (%)', ylabel='Model PART'); plt.show()

## 6. Apa perbedaan PART yang segera rusak dan yang tidak?
Kelompok **positif** mengalami kerusakan dalam 30 hari; kelompok **negatif** tidak. Semua data positif digunakan, sedangkan data negatif diambil sebagai sampel tetap agar laporan tidak perlu memuat lebih dari satu juta baris.

In [ ]:
sample = query("""SELECT target_failure_30d, days_since_installation, prior_failure_count, prior_corrective_count, prior_relocation_count, prior_distinct_places FROM analytics.item_observation_30d WHERE is_training_eligible AND (target_failure_30d OR MOD(ABS(HASHTEXT(installation_cycle_id || observation_date::text)::bigint), 100) < 3)""")
numeric_features = ['days_since_installation', 'prior_failure_count', 'prior_corrective_count', 'prior_relocation_count', 'prior_distinct_places']
sample[numeric_features] = sample[numeric_features].apply(pd.to_numeric, errors='coerce')
feature_summary = sample.groupby('target_failure_30d').agg(observations=('days_since_installation', 'size'), mean_age_days=('days_since_installation', 'mean'), median_age_days=('days_since_installation', 'median'), avg_prior_failures=('prior_failure_count', 'mean'), avg_prior_corrective=('prior_corrective_count', 'mean'), avg_prior_relocations=('prior_relocation_count', 'mean'), avg_prior_places=('prior_distinct_places', 'mean')).round(2)
feature_summary.index = feature_summary.index.map({False: 'Tidak rusak dalam 30 hari', True: 'Rusak dalam 30 hari'})
display(feature_summary.rename(columns={'observations': 'Jumlah snapshot', 'mean_age_days': 'Rata-rata umur (hari)', 'median_age_days': 'Median umur (hari)', 'avg_prior_failures': 'Rata-rata kerusakan sebelumnya', 'avg_prior_corrective': 'Rata-rata corrective sebelumnya', 'avg_prior_relocations': 'Rata-rata perpindahan', 'avg_prior_places': 'Rata-rata jumlah lokasi'}))
plot_data = sample[sample.days_since_installation <= sample.days_since_installation.quantile(.99)].copy()
sns.boxplot(data=plot_data, x='target_failure_30d', y='days_since_installation', showfliers=False).set(title='Perbandingan umur PART berdasarkan hasil 30 hari', xlabel='Mengalami kerusakan dalam 30 hari', ylabel='Hari sejak dipasang'); plt.show()

## 7. Apakah ada kerusakan yang mungkin belum terhitung?
Status `RETURNED` saja tidak selalu berarti rusak. Preventive dismantle yang kemudian dikonfirmasi `BROKEN` atau `UNREPAIRABLE` sudah dihitung sebagai kerusakan. Bagian ini mencari sisa status rusak yang tidak mempunyai corrective maupun preventive dismantle yang dapat dijadikan awal cycle.

In [ ]:
label_gap_summary = query("""WITH candidate AS (SELECT c.installation_cycle_id, BOOL_OR(o.event_semantic = 'RETURN_FLOW') has_return, BOOL_OR(o.event_semantic = 'FAILURE_OUTCOME') has_failure_outcome FROM analytics.item_installation_cycle c JOIN analytics.item_journey_operational_timeline o ON o.item_identifier_clean = c.item_identifier_clean AND o.created_on > c.installed_on AND o.created_on <= c.cycle_end_on WHERE NOT c.has_observed_failure AND o.event_semantic IN ('RETURN_FLOW', 'FAILURE_OUTCOME') GROUP BY c.installation_cycle_id) SELECT COUNT(*) candidate_cycles, COUNT(*) FILTER (WHERE has_return) returned_cycles, COUNT(*) FILTER (WHERE has_failure_outcome) explicit_failure_outcome_cycles FROM candidate""")
display(label_gap_summary.rename(columns={'candidate_cycles': 'Siklus yang perlu diperiksa', 'returned_cycles': 'Siklus dengan RETURNED', 'explicit_failure_outcome_cycles': 'Siklus dengan status rusak yang jelas'}))
label_gap_detail = query("""SELECT o.event_semantic, o.status_clean, COUNT(*) event_count, COUNT(DISTINCT c.installation_cycle_id) cycle_count FROM analytics.item_installation_cycle c JOIN analytics.item_journey_operational_timeline o ON o.item_identifier_clean = c.item_identifier_clean AND o.created_on > c.installed_on AND o.created_on <= c.cycle_end_on WHERE NOT c.has_observed_failure AND o.event_semantic IN ('RETURN_FLOW', 'FAILURE_OUTCOME') GROUP BY o.event_semantic, o.status_clean ORDER BY o.event_semantic, event_count DESC""")
display(label_gap_detail.rename(columns={'event_semantic': 'Kelompok kejadian', 'status_clean': 'Status', 'event_count': 'Jumlah kejadian', 'cycle_count': 'Jumlah siklus'}))
sns.barplot(data=label_gap_detail, y='status_clean', x='cycle_count', hue='event_semantic').set(title='Status lanjutan tanpa catatan corrective dismantle', xlabel='Jumlah siklus', ylabel='Status'); plt.show()

## 8. Apakah lokasi dan model tertentu lebih sering diikuti kerusakan?
Grafik pertama membandingkan lokasi terakhir PART pada tanggal snapshot. Yang dibandingkan adalah **persentase**, bukan hanya jumlah kerusakan, supaya lokasi dengan banyak PART tidak otomatis terlihat paling buruk. Hanya lokasi yang berhasil dicocokkan ke master lokasi, memiliki sedikitnya 50 PART, dan mempunyai 10 snapshot positif yang ditampilkan. Grafik kedua membandingkan kombinasi model dan lokasi; setiap kombinasi harus memiliki sedikitnya 20 PART dan 10 snapshot positif. Warna semakin gelap berarti persentase positif semakin tinggi. Hasil ini menunjukkan hubungan, bukan bukti bahwa lokasi menyebabkan kerusakan.

In [ ]:
by_location = query("""SELECT last_place_clean, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items, COUNT(*) FILTER (WHERE target_failure_30d) positives, ROUND(100.0 * COUNT(*) FILTER (WHERE target_failure_30d) / COUNT(*), 4) positive_pct FROM analytics.item_observation_30d WHERE is_training_eligible AND is_location_feature_eligible GROUP BY last_place_clean HAVING COUNT(DISTINCT item_identifier_clean) >= 50 AND COUNT(*) FILTER (WHERE target_failure_30d) >= 10 ORDER BY positive_pct DESC LIMIT 20""")
by_location['positive_pct'] = pd.to_numeric(by_location['positive_pct'], errors='coerce')
display(by_location.rename(columns={'last_place_clean': 'Lokasi terakhir', 'observations': 'Jumlah snapshot', 'items': 'Jumlah PART', 'positives': 'Rusak dalam 30 hari', 'positive_pct': 'Persentase positif'}))
plt.figure(figsize=(9, 7)); sns.barplot(data=by_location, y='last_place_clean', x='positive_pct', color='steelblue').set(title='Persentase kerusakan 30 hari menurut lokasi terakhir', xlabel='Persentase positif (%)', ylabel='Lokasi terakhir'); plt.tight_layout(); plt.show()
model_location = query("""SELECT item_model_code_clean, last_place_clean, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items, COUNT(*) FILTER (WHERE target_failure_30d) positives, ROUND(100.0 * COUNT(*) FILTER (WHERE target_failure_30d) / COUNT(*), 4) positive_pct FROM analytics.item_observation_30d WHERE is_training_eligible AND is_location_feature_eligible GROUP BY 1, 2 HAVING COUNT(DISTINCT item_identifier_clean) >= 20 AND COUNT(*) FILTER (WHERE target_failure_30d) >= 10 ORDER BY positives DESC""")
model_location['positive_pct'] = pd.to_numeric(model_location['positive_pct'], errors='coerce')
display(model_location.sort_values('positive_pct', ascending=False).head(20).rename(columns={'item_model_code_clean': 'Model PART', 'last_place_clean': 'Lokasi terakhir', 'observations': 'Jumlah snapshot', 'items': 'Jumlah PART', 'positives': 'Rusak dalam 30 hari', 'positive_pct': 'Persentase positif'}))
heatmap_models = model_location.groupby('item_model_code_clean')['positives'].sum().nlargest(10).index
heatmap_locations = model_location.groupby('last_place_clean')['positives'].sum().nlargest(15).index
heatmap_source = model_location[model_location['item_model_code_clean'].isin(heatmap_models) & model_location['last_place_clean'].isin(heatmap_locations)]
heatmap_data = heatmap_source.pivot(index='item_model_code_clean', columns='last_place_clean', values='positive_pct')
plt.figure(figsize=(18, 7)); sns.heatmap(heatmap_data, cmap='YlOrRd', annot=True, fmt='.2f', linewidths=.4, cbar_kws={'label': 'Persentase positif (%)'}); plt.title('Persentase kerusakan menurut kombinasi model dan lokasi'); plt.xlabel('Lokasi terakhir'); plt.ylabel('Model PART'); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

## 9. Bagaimana data dibagi untuk membuat dan menguji model?
Data lama dipakai untuk belajar, sedangkan data yang lebih baru dipakai untuk menguji apakah model tetap bekerja pada masa depan. Pembagian acak tidak digunakan karena cara pencatatan berubah sejak 2025. Data 2026 baru tersedia sampai 3 Agustus 2026, sehingga jumlah tahun 2026 belum mewakili satu tahun penuh. Laporan ini belum melatih model.

In [ ]:
splits = query("""SELECT CASE WHEN observation_on < DATE '2025-01-01' THEN 'TRAIN_2013_2024' WHEN observation_on < DATE '2026-01-01' THEN 'VALIDATION_2025' ELSE 'TEST_2026' END split, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items, COUNT(*) FILTER (WHERE target_failure_30d) positives, ROUND(100.0 * COUNT(*) FILTER (WHERE target_failure_30d) / COUNT(*), 4) positive_pct FROM analytics.item_observation_30d WHERE is_training_eligible GROUP BY 1 ORDER BY MIN(observation_on)""")
display(splits.replace({'TRAIN_2013_2024': 'Data belajar: 2013-2024', 'VALIDATION_2025': 'Pemeriksaan awal: 2025', 'TEST_2026': 'Pengujian akhir: 2026'}).rename(columns={'split': 'Kegunaan data', 'observations': 'Jumlah snapshot', 'items': 'Jumlah PART', 'positives': 'Rusak dalam 30 hari', 'positive_pct': 'Persentase positif'}))

## 10. Apakah snapshot 7 hari lebih baik daripada 30 hari?
Keduanya tetap mencari kerusakan dalam 30 hari. Snapshot selalu dimulai pada tanggal pemasangan, sehingga kerusakan cepat di antara jadwal tidak hilang. Snapshot 7 hari memberi lebih banyak baris training, tetapi satu failure dapat menghasilkan beberapa peringatan positif. Jarak snapshot untuk membuat data training juga berbeda dari jadwal pemakaian model: model yang dilatih dengan snapshot 30 hari tetap dapat dijalankan setiap hari atau setiap ada event baru.

In [ ]:
cadence = query('SELECT * FROM analytics.eda_snapshot_cadence_comparison ORDER BY cadence_days')
cadence_numeric = ['all_snapshots', 'eligible_snapshots', 'incomplete_followup_snapshots', 'positive_snapshots', 'positive_percentage', 'failure_cycles', 'captured_failure_cycles', 'uncaptured_failure_cycles', 'average_positive_snapshots_per_failure']
cadence[cadence_numeric] = cadence[cadence_numeric].apply(pd.to_numeric, errors='coerce')
display(cadence.rename(columns={'cadence_days': 'Jarak snapshot (hari)', 'all_snapshots': 'Semua snapshot', 'eligible_snapshots': 'Layak digunakan', 'incomplete_followup_snapshots': 'Follow-up belum lengkap', 'positive_snapshots': 'Snapshot positif', 'positive_percentage': 'Persentase positif', 'failure_cycles': 'Cycle dengan failure', 'captured_failure_cycles': 'Failure tertangkap', 'uncaptured_failure_cycles': 'Failure tidak tertangkap', 'average_positive_snapshots_per_failure': 'Rata-rata peringatan per failure'}))
sns.barplot(data=cadence, x='cadence_days', y='average_positive_snapshots_per_failure', color='steelblue').set(title='Berapa kali satu failure mendapat peringatan?', xlabel='Jarak snapshot (hari)', ylabel='Rata-rata snapshot positif'); plt.show()
unit_comparison = query('SELECT * FROM analytics.eda_failure_unit_comparison ORDER BY analysis_unit')
display(unit_comparison.rename(columns={'analysis_unit': 'Cara menghitung', 'population_count': 'Jumlah yang diperiksa', 'positive_count': 'Jumlah positif', 'positive_percentage': 'Persentase positif', 'explanation': 'Arti'}))
fast_failure = query("""SELECT COUNT(*) FILTER (WHERE failure_onset_on - installed_on <= INTERVAL '1 day') failure_le_1d, COUNT(*) FILTER (WHERE failure_onset_on > installed_on AND failure_onset_on - installed_on <= INTERVAL '1 day') captured_from_first_snapshot FROM analytics.item_installation_cycle WHERE is_initial_model_cohort AND has_observed_failure""")
display(Markdown(f"Ada **{int(fast_failure.failure_le_1d.iloc[0])}** failure dalam maksimal satu hari setelah pemasangan, dan **{int(fast_failure.captured_from_first_snapshot.iloc[0])}** semuanya tertangkap dari snapshot pertama."))

## 11. Mana failure yang mungkin masih berjalan dan mana yang kemungkinan kehilangan histori?
Failure tanpa RETURN atau proses repair dibagi menurut jaraknya dari tanggal terakhir data. Kasus 0-30 hari kemungkinan masih berjalan; 31-180 hari perlu dipantau; lebih dari 180 hari lebih mungkin mempunyai histori yang tidak lengkap. Kandidat `RETURNED → BROKEN` tanpa dismantle tetap dipisahkan untuk review dan belum menjadi label model.

In [ ]:
incomplete = query("""SELECT * FROM analytics.eda_incomplete_failure_summary ORDER BY CASE followup_review_group WHEN 'LIKELY_ONGOING_0_30D' THEN 1 WHEN 'REVIEW_31_180D' THEN 2 ELSE 3 END""")
incomplete_labels = {'LIKELY_ONGOING_0_30D': 'Kemungkinan masih berjalan (0-30 hari)', 'REVIEW_31_180D': 'Perlu dipantau (31-180 hari)', 'LIKELY_HISTORY_GAP_GT_180D': 'Kemungkinan histori hilang (>180 hari)'}
incomplete_display = incomplete.assign(followup_review_group=incomplete['followup_review_group'].replace(incomplete_labels)).rename(columns={'followup_review_group': 'Kelompok pemeriksaan', 'failure_count': 'Jumlah failure', 'item_count': 'Jumlah PART', 'earliest_failure_date': 'Tanggal paling awal', 'latest_failure_date': 'Tanggal paling akhir'})
display(incomplete_display)
sns.barplot(data=incomplete_display, y='Kelompok pemeriksaan', x='Jumlah failure', color='darkorange').set(title='Failure tanpa catatan proses lanjutan', xlabel='Jumlah failure', ylabel=''); plt.show()
missing_onset = query('SELECT item_model_code_clean, installed_on::date installed_on, installed_place_clean, outcome_on::date outcome_on, outcome_status, suggested_label FROM analytics.failure_outcome_missing_onset_review ORDER BY outcome_on')
display(missing_onset.rename(columns={'item_model_code_clean': 'Model PART', 'installed_on': 'Tanggal dipasang', 'installed_place_clean': 'Lokasi pemasangan', 'outcome_on': 'Tanggal status rusak', 'outcome_status': 'Status rusak', 'suggested_label': 'Saran label review'}))

## 12. Apakah ada fitur redundan dan fitur mana yang punya predictive power awal?
Pearson memeriksa hubungan linear, sedangkan Spearman memeriksa apakah dua fitur cenderung naik bersama walaupun hubungannya tidak lurus. Sampel korelasi diambil merata dari semua snapshot agar positive tidak dibesarkan secara buatan. Pasangan dengan |Spearman| minimal 0,80 ditandai redundan untuk review, tetapi tidak langsung dibuang. Information Value (IV) kemudian dipakai sebagai screening univariat terhadap target; IV bukan feature importance model final dan harus dikonfirmasi dengan split waktu.

In [ ]:
candidate_numeric_features = ['days_since_installation','days_since_last_event','days_since_last_failure','days_since_last_corrective','days_at_last_location','total_prior_events','prior_failure_count','prior_corrective_count','prior_relocation_count','prior_preventive_count','prior_repair_process_count','prior_events_30d','prior_events_90d','prior_events_180d','prior_corrective_30d','prior_corrective_90d','prior_corrective_180d','prior_preventive_90d','prior_failure_365d','prior_distinct_places']
feature_sql = ', '.join(candidate_numeric_features)
correlation_sample = query(f"SELECT target_failure_30d, {feature_sql} FROM analytics.item_observation_30d WHERE is_training_eligible AND MOD(ABS(HASHTEXT(installation_cycle_id || observation_date::text)::bigint), 20)=0")
correlation_sample[candidate_numeric_features] = correlation_sample[candidate_numeric_features].apply(pd.to_numeric, errors='coerce')
correlation_features = [c for c in candidate_numeric_features if correlation_sample[c].nunique(dropna=True) > 1]
pearson_corr = correlation_sample[correlation_features].corr(method='pearson')
spearman_corr = correlation_sample[correlation_features].corr(method='spearman')
fig, axes = plt.subplots(1, 2, figsize=(22, 9)); sns.heatmap(pearson_corr, cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=axes[0]); axes[0].set_title('Korelasi Pearson (hubungan linear)'); sns.heatmap(spearman_corr, cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=axes[1]); axes[1].set_title('Korelasi Spearman (urutan/monotonik)'); plt.tight_layout(); plt.show()
upper_mask = np.triu(np.ones(spearman_corr.shape, dtype=bool), k=1)
redundant_pairs = spearman_corr.abs().where(upper_mask).stack().reset_index()
redundant_pairs.columns = ['feature_1','feature_2','abs_spearman']
redundant_pairs = redundant_pairs.loc[redundant_pairs.abs_spearman.ge(0.80)].sort_values('abs_spearman', ascending=False)
redundant_pairs['keputusan'] = np.where(redundant_pairs.abs_spearman.ge(0.95), 'Hampir duplikat: prioritaskan satu setelah validasi waktu', 'Redundan tinggi: uji salah satu/interaksi, jangan masukkan buta')
display(redundant_pairs.rename(columns={'feature_1':'Fitur pertama','feature_2':'Fitur kedua','abs_spearman':'|Spearman|','keputusan':'Tindakan'}))
iv_sample = query(f"SELECT target_failure_30d, {feature_sql} FROM analytics.item_observation_30d WHERE is_training_eligible AND (target_failure_30d OR MOD(ABS(HASHTEXT(installation_cycle_id || observation_date::text)::bigint),100)<3)")
iv_sample[candidate_numeric_features] = iv_sample[candidate_numeric_features].apply(pd.to_numeric, errors='coerce')
def calculate_information_value(frame, feature, target='target_failure_30d'):
    work = frame[[feature, target]].copy()
    nonmissing = work[feature].dropna()
    if nonmissing.nunique() > 10:
        bins = pd.Series('MISSING', index=work.index, dtype='object')
        try:
            bins.loc[nonmissing.index] = pd.qcut(nonmissing, q=10, duplicates='drop').astype(str)
        except ValueError:
            bins.loc[nonmissing.index] = nonmissing.astype(str)
    else:
        bins = work[feature].astype('string').fillna('MISSING')
    table = pd.crosstab(bins, work[target].astype(bool))
    negative = table.get(False, pd.Series(0, index=table.index)).astype(float) + 0.5
    positive = table.get(True, pd.Series(0, index=table.index)).astype(float) + 0.5
    negative_dist, positive_dist = negative / negative.sum(), positive / positive.sum()
    return float(((positive_dist-negative_dist) * np.log(positive_dist/negative_dist)).sum())
iv_result = pd.DataFrame({'feature': candidate_numeric_features, 'information_value': [calculate_information_value(iv_sample, f) for f in candidate_numeric_features]})
iv_result['screening'] = pd.cut(iv_result.information_value, bins=[-np.inf,.02,.10,.30,.50,np.inf], labels=['Sangat lemah','Lemah','Sedang','Kuat','Sangat kuat: cek leakage/drift'])
iv_result = iv_result.sort_values('information_value', ascending=False)
display(iv_result.rename(columns={'feature':'Fitur','information_value':'Information Value (IV)','screening':'Interpretasi awal'}))
plt.figure(figsize=(9, 7)); sns.barplot(data=iv_result, y='feature', x='information_value', color='darkcyan').set(title='Predictive power awal per fitur (IV univariat)', xlabel='Information Value', ylabel='Fitur'); plt.tight_layout(); plt.show()

## 13. Apakah distribusi fitur dan risiko tetap stabil terhadap waktu?
Grafik bulanan memeriksa perubahan rata-rata fitur operasional utama. Population Stability Index (PSI) membandingkan distribusi 2025 dan 2026 terhadap referensi 2024: PSI <0,10 relatif stabil; 0,10-0,25 perlu dipantau; dan >=0,25 menunjukkan drift besar. Tahun 2026 masih parsial sampai cutoff sehingga interpretasinya harus hati-hati. Heatmap setelahnya tetap memeriksa perubahan positive rate model dan lokasi.

In [ ]:
monthly_feature_stability = query("SELECT * FROM analytics.eda_feature_stability_monthly WHERE observation_month_start >= DATE '2024-01-01' ORDER BY feature_name, observation_month_start")
monthly_feature_stability['observation_month_start'] = pd.to_datetime(monthly_feature_stability.observation_month_start)
for col in ['mean_value','median_value','p90_value','missing_percentage']: monthly_feature_stability[col] = pd.to_numeric(monthly_feature_stability[col], errors='coerce')
monthly_plot = monthly_feature_stability[monthly_feature_stability.feature_name.isin(['prior_events_90d','prior_corrective_90d','prior_failure_365d','days_since_installation'])]
g = sns.relplot(data=monthly_plot, x='observation_month_start', y='mean_value', col='feature_name', col_wrap=2, kind='line', marker='o', facet_kws={'sharey':False}, height=3.4, aspect=1.6); g.set_axis_labels('Bulan snapshot','Rata-rata fitur'); g.set_titles('{col_name}'); g.fig.suptitle('Perubahan rata-rata fitur utama per bulan', y=1.03); plt.show()
drift_sample = query(f"SELECT observation_year, {feature_sql} FROM analytics.item_observation_30d WHERE is_training_eligible AND observation_on>=DATE '2024-01-01' AND MOD(ABS(HASHTEXT(installation_cycle_id || observation_date::text)::bigint),10)=0")
drift_sample[candidate_numeric_features] = drift_sample[candidate_numeric_features].apply(pd.to_numeric, errors='coerce')
def population_stability_index(reference, current):
    reference, current = pd.Series(reference), pd.Series(current)
    if reference.dropna().nunique() <= 10:
        ref_group = reference.astype('object').where(reference.notna(), 'MISSING').astype(str)
        cur_group = current.astype('object').where(current.notna(), 'MISSING').astype(str)
    else:
        internal = np.unique(reference.dropna().quantile(np.linspace(0,1,11)).to_numpy())[1:-1]
        edges = np.r_[-np.inf, internal, np.inf]
        ref_group = pd.cut(reference, edges, include_lowest=True, duplicates='drop').astype('string').fillna('MISSING')
        cur_group = pd.cut(current, edges, include_lowest=True, duplicates='drop').astype('string').fillna('MISSING')
    categories = sorted(set(ref_group) | set(cur_group))
    ref_pct = ref_group.value_counts(normalize=True).reindex(categories, fill_value=0).to_numpy(float)
    cur_pct = cur_group.value_counts(normalize=True).reindex(categories, fill_value=0).to_numpy(float)
    ref_pct, cur_pct = np.clip(ref_pct, 1e-4, None), np.clip(cur_pct, 1e-4, None)
    ref_pct, cur_pct = ref_pct/ref_pct.sum(), cur_pct/cur_pct.sum()
    return float(np.sum((cur_pct-ref_pct)*np.log(cur_pct/ref_pct)))
psi_rows = []
for feature in candidate_numeric_features:
    reference = drift_sample.loc[drift_sample.observation_year.eq(2024), feature]
    for year in [2025, 2026]:
        current = drift_sample.loc[drift_sample.observation_year.eq(year), feature]
        psi_rows.append((feature, year, population_stability_index(reference, current)))
psi_result = pd.DataFrame(psi_rows, columns=['feature','comparison_year','psi'])
psi_result['drift_status'] = pd.cut(psi_result.psi, bins=[-np.inf,.10,.25,np.inf], labels=['Relatif stabil','Perlu dipantau','Drift besar'])
display(psi_result.sort_values(['comparison_year','psi'], ascending=[True,False]).rename(columns={'feature':'Fitur','comparison_year':'Tahun dibanding 2024','psi':'PSI','drift_status':'Status drift'}))
psi_heatmap = psi_result.pivot(index='feature', columns='comparison_year', values='psi')
plt.figure(figsize=(7, 8)); sns.heatmap(psi_heatmap, annot=True, fmt='.3f', cmap='YlOrRd', vmin=0, cbar_kws={'label':'PSI terhadap 2024'}); plt.title('Drift distribusi fitur terhadap referensi 2024'); plt.xlabel('Tahun'); plt.ylabel('Fitur'); plt.tight_layout(); plt.show()
model_stability = query("""WITH top_model AS (SELECT item_model_code_clean FROM analytics.item_observation_30d WHERE is_training_eligible GROUP BY 1 ORDER BY COUNT(*) FILTER (WHERE target_failure_30d) DESC LIMIT 10) SELECT CASE WHEN observation_on < DATE '2025-01-01' THEN '2013-2024' WHEN observation_on < DATE '2026-01-01' THEN '2025' ELSE '2026' END period, item_model_code_clean, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items, COUNT(*) FILTER (WHERE target_failure_30d) positives, ROUND(100.0 * COUNT(*) FILTER (WHERE target_failure_30d) / COUNT(*), 4) positive_pct FROM analytics.item_observation_30d JOIN top_model USING (item_model_code_clean) WHERE is_training_eligible GROUP BY 1, 2 HAVING COUNT(DISTINCT item_identifier_clean) >= 20""")
model_stability['positive_pct'] = pd.to_numeric(model_stability['positive_pct'], errors='coerce')
model_stability_heatmap = model_stability.pivot(index='item_model_code_clean', columns='period', values='positive_pct')
plt.figure(figsize=(7, 7)); sns.heatmap(model_stability_heatmap, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': 'Persentase positif (%)'}); plt.title('Perubahan risiko model PART menurut periode'); plt.xlabel('Periode'); plt.ylabel('Model PART'); plt.tight_layout(); plt.show()
location_stability = query("""WITH top_location AS (SELECT last_place_clean FROM analytics.item_observation_30d WHERE is_training_eligible AND is_location_feature_eligible GROUP BY 1 ORDER BY COUNT(*) FILTER (WHERE target_failure_30d) DESC LIMIT 10) SELECT CASE WHEN observation_on < DATE '2025-01-01' THEN '2013-2024' WHEN observation_on < DATE '2026-01-01' THEN '2025' ELSE '2026' END period, last_place_clean, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items, COUNT(*) FILTER (WHERE target_failure_30d) positives, ROUND(100.0 * COUNT(*) FILTER (WHERE target_failure_30d) / COUNT(*), 4) positive_pct FROM analytics.item_observation_30d JOIN top_location USING (last_place_clean) WHERE is_training_eligible AND is_location_feature_eligible GROUP BY 1, 2 HAVING COUNT(DISTINCT item_identifier_clean) >= 30""")
location_stability['positive_pct'] = pd.to_numeric(location_stability['positive_pct'], errors='coerce')
location_stability_heatmap = location_stability.pivot(index='last_place_clean', columns='period', values='positive_pct')
plt.figure(figsize=(7, 8)); sns.heatmap(location_stability_heatmap, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': 'Persentase positif (%)'}); plt.title('Perubahan risiko lokasi menurut periode'); plt.xlabel('Periode'); plt.ylabel('Lokasi terakhir'); plt.tight_layout(); plt.show()

## 14. Data mana yang masih perlu diperiksa manual?
Nilai di bawah ini tidak otomatis salah. Daftar ini menunjukkan kelompok yang perlu diambil sampelnya sebelum keputusan akhir modeling.

In [ ]:
outliers = query('SELECT * FROM analytics.eda_outlier_summary ORDER BY affected_count DESC')
outlier_labels = {'OPERATIONAL_GAP_GT_10Y': 'Jarak kegiatan lebih dari 10 tahun', 'ZERO_OR_NEGATIVE_DURATION_CYCLE': 'Cycle tanpa durasi positif', 'FAILURE_NOT_PRECEDED_BY_INSTALLED': 'Failure tidak langsung didahului INSTALLED', 'ITEM_WITH_5_PLUS_FAILURES': 'PART dengan minimal 5 failure', 'JOURNEY_MODEL_INCONSISTENT': 'Model journey tidak konsisten', 'INVALID_OR_FUTURE_JOURNEY_DATE': 'Tanggal journey invalid/masa depan', 'SNAPSHOT_WITHOUT_MASTER_LOCATION': 'Snapshot tanpa lokasi master'}
outliers_display = outliers.assign(check_name=outliers['check_name'].replace(outlier_labels)).rename(columns={'check_name': 'Yang perlu diperiksa', 'affected_count': 'Jumlah terdampak', 'explanation': 'Alasan'})
display(outliers_display)

## 15. Audit kualitas data journal secara lengkap
Pemeriksaan sebelumnya berfokus pada dataset model. Bagian ini mengaudit sumber journey yang benar-benar dipakai: data kosong, lokasi non-master, tanggal invalid, duplikasi key, duplikasi seluruh isi log, dan tipe data tanggal. Nilai nol berarti pemeriksaan tersebut lolos.

In [ ]:
journal_quality = query('SELECT * FROM analytics.eda_journey_quality_summary ORDER BY check_order')
quality_labels = {'MISSING_ITEM_IDENTIFIER': 'Identifier item kosong', 'MISSING_ITEM_MODEL': 'Model item kosong', 'MISSING_ITEM_TYPE': 'Tipe item kosong', 'MISSING_ITEM_CATEGORY': 'Kategori item kosong', 'MISSING_CLIENT': 'Klien kosong', 'MISSING_LOCATION': 'Lokasi mentah kosong', 'LOCATION_NOT_IN_MASTER': 'Lokasi belum cocok setelah mapping', 'CLIENT_NOT_IN_MASTER': 'Klien belum cocok setelah mapping', 'MISSING_STATUS': 'Status kosong', 'MISSING_DATE': 'Tanggal kosong', 'INVALID_OR_FUTURE_DATE': 'Tanggal invalid/masa depan', 'DUPLICATE_JOURNEY_ID': 'journey_id duplikat', 'EXACT_LOG_DUPLICATE_EXTRA_ROWS': 'Baris tambahan dengan isi log identik', 'CREATED_ON_NOT_DATETIME': 'created_on bukan datetime'}
journal_quality['affected_count'] = pd.to_numeric(journal_quality['affected_count'], errors='coerce').fillna(0).astype(int)
journal_quality_display = journal_quality.assign(check_name=journal_quality['check_name'].replace(quality_labels)).rename(columns={'check_name': 'Pemeriksaan', 'affected_count': 'Jumlah terdampak', 'explanation': 'Arti'})
display(journal_quality_display[['Pemeriksaan', 'Jumlah terdampak', 'Arti']])
quality_nonzero = journal_quality_display[journal_quality_display['Jumlah terdampak'].gt(0)]
if not quality_nonzero.empty:
    plt.figure(figsize=(9, 4)); sns.barplot(data=quality_nonzero, y='Pemeriksaan', x='Jumlah terdampak', color='indianred'); plt.xscale('log'); plt.title('Masalah kualitas journal yang benar-benar ditemukan'); plt.xlabel('Jumlah terdampak (skala log)'); plt.ylabel(''); plt.tight_layout(); plt.show()
date_types = query("""SELECT table_schema, table_name, column_name, data_type FROM information_schema.columns WHERE (table_schema, table_name, column_name) IN (('journal','t_item_journey','created_on'), ('analytics','item_journey_clean','created_on')) ORDER BY table_schema""")
display(date_types.rename(columns={'table_schema': 'Layer', 'table_name': 'Tabel/view', 'column_name': 'Kolom', 'data_type': 'Tipe data'}))
duplicate_groups = query("""SELECT ARRAY_AGG(journey_id ORDER BY journey_id)::text journey_ids, item_model_code_clean, status_clean, activity_clean, created_on, place_clean, COUNT(*) row_count FROM analytics.item_journey_clean GROUP BY item_identifier_clean, created_on, item_category_clean, item_type_clean, item_model_code_clean, client_clean, ref_doc_code_clean, wo_type_clean, wo_code_clean, place_clean, activity_clean, status_clean, done_by_clean, remark HAVING COUNT(*) > 1 ORDER BY row_count DESC, created_on LIMIT 20""")
display(duplicate_groups.rename(columns={'journey_ids': 'journey_id yang perlu dibandingkan', 'item_model_code_clean': 'Model', 'status_clean': 'Status', 'activity_clean': 'Aktivitas', 'created_on': 'Waktu', 'place_clean': 'Lokasi mentah', 'row_count': 'Jumlah baris'}))
fuzzy_mapping = query('SELECT * FROM analytics.eda_fuzzy_mapping_review ORDER BY mapping_type, event_count DESC')
fuzzy_mapping['similarity_percentage'] = (pd.to_numeric(fuzzy_mapping.similarity_score, errors='coerce') * 100).round(2); fuzzy_mapping['margin_percentage'] = (pd.to_numeric(fuzzy_mapping.score_margin, errors='coerce') * 100).round(2)
display(fuzzy_mapping.rename(columns={'mapping_type': 'Jenis', 'source_value': 'Nilai sumber', 'canonical_value': 'Nilai canonical', 'best_candidate': 'Kandidat terbaik', 'mapping_method': 'Keputusan', 'similarity_percentage': 'Kemiripan (%)', 'margin_percentage': 'Selisih kandidat (%)', 'event_count': 'Event', 'item_count': 'Item'})[['Jenis','Nilai sumber','Nilai canonical','Kandidat terbaik','Keputusan','Kemiripan (%)','Selisih kandidat (%)','Event','Item']])

## 16. Analisis univariat item, lokasi, dan waktu
Analisis ini menghitung setiap variabel secara terpisah. Item berarti model yang muncul pada event operasional; lokasi hanya memakai nama yang cocok dengan master. Model/lokasi dengan aktivitas rendah tetap ditampilkan agar tidak tertutup oleh kelompok besar.

In [ ]:
item_activity = query('SELECT * FROM analytics.eda_item_activity_summary ORDER BY event_count DESC')
for col in ['event_count', 'item_count', 'installation_count', 'dismantle_count', 'failure_count']: item_activity[col] = pd.to_numeric(item_activity[col], errors='coerce')
top_items = item_activity.head(15)
rare_items = item_activity.sort_values(['event_count', 'item_count']).head(15)
display(top_items.rename(columns={'item_category_clean': 'Kategori', 'item_model_code_clean': 'Model', 'event_count': 'Event', 'item_count': 'Unit', 'installation_count': 'Installed', 'dismantle_count': 'Dismantled', 'failure_count': 'Failure', 'first_activity_date': 'Pertama', 'last_activity_date': 'Terakhir'}))
display(Markdown(f"Ada **{int(((item_activity.event_count <= 5) | (item_activity.item_count <= 1)).sum())}** model dengan maksimal lima event atau hanya satu unit. Kelompok kecil ini tidak aman dibandingkan berdasarkan persentase tanpa batas minimum sampel."))
display(rare_items[['item_category_clean','item_model_code_clean','event_count','item_count','first_activity_date','last_activity_date']].rename(columns={'item_category_clean': 'Kategori', 'item_model_code_clean': 'Model jarang', 'event_count': 'Event', 'item_count': 'Unit', 'first_activity_date': 'Pertama', 'last_activity_date': 'Terakhir'}))
plt.figure(figsize=(9, 6)); sns.barplot(data=top_items, y='item_model_code_clean', x='event_count', hue='item_category_clean', dodge=False); plt.title('15 model dengan aktivitas operasional terbanyak'); plt.xlabel('Jumlah event'); plt.ylabel('Model'); plt.tight_layout(); plt.show()
location_activity = query('SELECT * FROM analytics.eda_location_activity_summary ORDER BY event_count DESC')
for col in ['event_count', 'item_count', 'model_count', 'installation_count', 'dismantle_count', 'failure_count']: location_activity[col] = pd.to_numeric(location_activity[col], errors='coerce')
top_locations = location_activity.head(15); rare_locations = location_activity.sort_values('event_count').head(15)
display(top_locations.rename(columns={'place_canonical_clean': 'Lokasi', 'event_count': 'Event', 'item_count': 'Unit', 'model_count': 'Model', 'installation_count': 'Installed', 'dismantle_count': 'Dismantled', 'failure_count': 'Failure', 'first_activity_date': 'Pertama', 'last_activity_date': 'Terakhir'}))
display(rare_locations[['place_canonical_clean','event_count','item_count','model_count','first_activity_date','last_activity_date']].rename(columns={'place_canonical_clean': 'Lokasi dengan aktivitas rendah', 'event_count': 'Event', 'item_count': 'Unit', 'model_count': 'Model', 'first_activity_date': 'Pertama', 'last_activity_date': 'Terakhir'}))
plt.figure(figsize=(9, 6)); sns.barplot(data=top_locations, y='place_canonical_clean', x='event_count', color='seagreen'); plt.title('15 lokasi dengan aktivitas operasional terbanyak'); plt.xlabel('Jumlah event'); plt.ylabel('Lokasi'); plt.tight_layout(); plt.show()
monthly_activity = query("SELECT activity_month, SUM(event_count) event_count, SUM(installation_count) installation_count, SUM(dismantle_count) dismantle_count, SUM(failure_count) failure_count FROM analytics.eda_activity_calendar_summary GROUP BY activity_month ORDER BY activity_month")
monthly_activity['activity_month'] = pd.to_datetime(monthly_activity['activity_month']); monthly_activity[['event_count','installation_count','dismantle_count','failure_count']] = monthly_activity[['event_count','installation_count','dismantle_count','failure_count']].apply(pd.to_numeric)
monthly_long = monthly_activity.melt(id_vars='activity_month', value_vars=['installation_count','dismantle_count'], var_name='activity_type', value_name='count')
plt.figure(figsize=(12, 4)); sns.lineplot(data=monthly_long, x='activity_month', y='count', hue='activity_type'); plt.title('Tren pemasangan dan dismantle per bulan'); plt.xlabel('Bulan'); plt.ylabel('Jumlah event'); plt.tight_layout(); plt.show()
weekday_activity = query("SELECT iso_day_of_week, is_weekend, SUM(event_count) event_count, SUM(installation_count) installation_count, SUM(dismantle_count) dismantle_count FROM analytics.eda_activity_calendar_summary GROUP BY iso_day_of_week, is_weekend ORDER BY iso_day_of_week")
weekday_labels = {1:'Senin',2:'Selasa',3:'Rabu',4:'Kamis',5:'Jumat',6:'Sabtu',7:'Minggu'}; weekday_activity['Hari'] = weekday_activity.iso_day_of_week.map(weekday_labels); weekday_activity['event_count'] = pd.to_numeric(weekday_activity.event_count)
display(weekday_activity.rename(columns={'is_weekend': 'Akhir pekan', 'event_count': 'Event', 'installation_count': 'Installed', 'dismantle_count': 'Dismantled'})[['Hari','Akhir pekan','Event','Installed','Dismantled']])
plt.figure(figsize=(9, 4)); sns.barplot(data=weekday_activity, x='Hari', y='event_count', hue='is_weekend', dodge=False); plt.title('Aktivitas berdasarkan hari dalam minggu'); plt.xlabel('Hari'); plt.ylabel('Jumlah event'); plt.tight_layout(); plt.show()

## 17. Hubungan item, lokasi, dan waktu
Bagian ini menjawab apakah model PART hanya dipasang di satu lokasi atau tersebar, lalu melihat kombinasi model-lokasi yang paling sering serta perubahan pemasangan model/lokasi dalam 36 bulan terakhir. Nilai adalah jumlah installation, bukan positive rate failure.

In [ ]:
model_location_scope = query("""WITH scope AS (SELECT item_model_code_clean, COUNT(*) location_count, SUM(installation_count) installations, SUM(item_count) item_location_pairs FROM analytics.eda_item_location_installation_summary GROUP BY item_model_code_clean) SELECT CASE WHEN location_count=1 THEN 'Hanya 1 lokasi' WHEN location_count<=5 THEN '2-5 lokasi' WHEN location_count<=20 THEN '6-20 lokasi' ELSE '>20 lokasi' END location_scope, COUNT(*) model_count, SUM(installations) installations FROM scope GROUP BY 1 ORDER BY MIN(location_count)""")
display(model_location_scope.rename(columns={'location_scope': 'Sebaran model', 'model_count': 'Jumlah model', 'installations': 'Jumlah pemasangan'}))
item_location_matrix = query("""WITH top_model AS (SELECT item_model_code_clean FROM analytics.eda_item_location_installation_summary GROUP BY 1 ORDER BY SUM(installation_count) DESC LIMIT 12), top_location AS (SELECT place_canonical_clean FROM analytics.eda_item_location_installation_summary GROUP BY 1 ORDER BY SUM(installation_count) DESC LIMIT 12) SELECT s.item_model_code_clean, s.place_canonical_clean, s.installation_count FROM analytics.eda_item_location_installation_summary s JOIN top_model m USING (item_model_code_clean) JOIN top_location l USING (place_canonical_clean)""")
item_location_matrix['installation_count'] = pd.to_numeric(item_location_matrix.installation_count); item_location_pivot = item_location_matrix.pivot(index='item_model_code_clean', columns='place_canonical_clean', values='installation_count').fillna(0)
plt.figure(figsize=(13, 8)); sns.heatmap(item_location_pivot, cmap='Blues', annot=True, fmt='.0f'); plt.title('Jumlah pemasangan: 12 model dan 12 lokasi paling aktif'); plt.xlabel('Lokasi'); plt.ylabel('Model PART'); plt.tight_layout(); plt.show()
recent_model_trend = query("""WITH boundary AS (SELECT MAX(created_on) cutoff FROM analytics.item_journey_operational_timeline), recent AS (SELECT DATE_TRUNC('month', o.created_on)::date activity_month, o.item_model_code_clean FROM analytics.item_journey_operational_timeline o CROSS JOIN boundary b WHERE o.item_category_clean='PART' AND o.status_clean='INSTALLED' AND o.created_on>b.cutoff-INTERVAL '36 months'), top_model AS (SELECT item_model_code_clean FROM recent GROUP BY 1 ORDER BY COUNT(*) DESC LIMIT 5) SELECT activity_month, item_model_code_clean, COUNT(*) installation_count FROM recent JOIN top_model USING (item_model_code_clean) GROUP BY 1,2 ORDER BY 1,2""")
recent_model_trend['activity_month'] = pd.to_datetime(recent_model_trend.activity_month); recent_model_trend['installation_count'] = pd.to_numeric(recent_model_trend.installation_count)
plt.figure(figsize=(12, 5)); sns.lineplot(data=recent_model_trend, x='activity_month', y='installation_count', hue='item_model_code_clean', marker='o'); plt.title('Pemasangan lima model teraktif dalam 36 bulan terakhir'); plt.xlabel('Bulan'); plt.ylabel('Jumlah pemasangan'); plt.tight_layout(); plt.show()
recent_location_trend = query("""WITH boundary AS (SELECT MAX(created_on) cutoff FROM analytics.item_journey_operational_timeline), recent AS (SELECT DATE_TRUNC('month', o.created_on)::date activity_month, o.place_canonical_clean FROM analytics.item_journey_operational_timeline o CROSS JOIN boundary b WHERE o.status_clean='INSTALLED' AND o.place_canonical_clean IS NOT NULL AND o.created_on>b.cutoff-INTERVAL '36 months'), top_location AS (SELECT place_canonical_clean FROM recent GROUP BY 1 ORDER BY COUNT(*) DESC LIMIT 8) SELECT activity_month, place_canonical_clean, COUNT(*) installation_count FROM recent JOIN top_location USING (place_canonical_clean) GROUP BY 1,2 ORDER BY 1,2""")
recent_location_trend['installation_count'] = pd.to_numeric(recent_location_trend.installation_count); recent_location_pivot = recent_location_trend.pivot(index='place_canonical_clean', columns='activity_month', values='installation_count').fillna(0)
plt.figure(figsize=(15, 6)); sns.heatmap(recent_location_pivot, cmap='YlOrRd'); plt.title('Kepadatan pemasangan delapan lokasi teraktif dalam 36 bulan terakhir'); plt.xlabel('Bulan'); plt.ylabel('Lokasi'); plt.tight_layout(); plt.show()

## 18. Lifecycle: berapa lama PART berada di lokasi sebelum dismantle?
Durasi dihitung dari INSTALLED sampai DISMANTLED pertama sebelum pemasangan berikutnya. Statistik lokasi hanya memakai pasangan yang lokasi installation dan dismantle-nya sama serta cocok dengan master. Cycle tanpa dismantle tidak dianggap berdurasi nol karena akhir lifecycle-nya belum diketahui.

In [ ]:
lifecycle_overall = query("""SELECT COUNT(*) installations, COUNT(*) FILTER (WHERE has_next_dismantle) with_next_dismantle, COUNT(*) FILTER (WHERE NOT has_next_dismantle) no_dismantle_before_next_install_or_cutoff, COUNT(*) FILTER (WHERE is_same_location) same_location, COUNT(*) FILTER (WHERE is_location_mismatch) location_mismatch, COUNT(*) FILTER (WHERE has_next_dismantle AND (installed_place_clean IS NULL OR dismantled_place_clean IS NULL)) location_unavailable, ROUND(AVG(days_installed_to_dismantle) FILTER (WHERE is_same_location)::numeric,2) average_days, ROUND(PERCENTILE_CONT(.5) WITHIN GROUP (ORDER BY days_installed_to_dismantle) FILTER (WHERE is_same_location)::numeric,2) median_days, ROUND(PERCENTILE_CONT(.9) WITHIN GROUP (ORDER BY days_installed_to_dismantle) FILTER (WHERE is_same_location)::numeric,2) p90_days, COUNT(*) FILTER (WHERE days_installed_to_dismantle<0) negative_duration FROM analytics.eda_location_lifecycle_detail""")
display(lifecycle_overall.rename(columns={'installations': 'Pemasangan PART', 'with_next_dismantle': 'Memiliki dismantle berikutnya', 'no_dismantle_before_next_install_or_cutoff': 'Belum ada dismantle yang dapat dipasangkan', 'same_location': 'Lokasi installation=dismantle', 'location_mismatch': 'Lokasi berbeda', 'location_unavailable': 'Lokasi tidak tersedia', 'average_days': 'Rata-rata hari', 'median_days': 'Median hari', 'p90_days': 'Persentil 90 hari', 'negative_duration': 'Durasi negatif'}))
lifecycle_location = query("SELECT * FROM analytics.eda_location_lifecycle_summary WHERE matched_lifecycle_count>=20 ORDER BY median_days DESC")
for col in ['installation_count','matched_lifecycle_count','average_days','median_days','p90_days']: lifecycle_location[col] = pd.to_numeric(lifecycle_location[col], errors='coerce')
display(lifecycle_location.rename(columns={'installed_place_clean': 'Lokasi pemasangan', 'installation_count': 'Semua pemasangan', 'matched_lifecycle_count': 'Lifecycle lokasi sama', 'average_days': 'Rata-rata hari', 'median_days': 'Median hari', 'p90_days': 'Persentil 90 hari'}).head(20))
lifecycle_plot = lifecycle_location.sort_values('matched_lifecycle_count', ascending=False).head(15).sort_values('median_days')
plt.figure(figsize=(9, 6)); sns.barplot(data=lifecycle_plot, y='installed_place_clean', x='median_days', color='slateblue'); plt.title('Median hari dari installed sampai dismantle pada lokasi yang sama'); plt.xlabel('Median hari'); plt.ylabel('Lokasi'); plt.tight_layout(); plt.show()
lifecycle_duration = query("SELECT days_installed_to_dismantle FROM analytics.eda_location_lifecycle_detail WHERE is_same_location")['days_installed_to_dismantle'].astype(float)
lifecycle_clip = lifecycle_duration.clip(upper=lifecycle_duration.quantile(.99)); plt.figure(figsize=(9, 4)); sns.histplot(lifecycle_clip, bins=40); plt.title('Distribusi lifecycle lokasi sama (dipotong pada persentil 99 untuk visual)'); plt.xlabel('Hari installed sampai dismantle'); plt.ylabel('Jumlah lifecycle'); plt.tight_layout(); plt.show()

## 19. Anomali, keputusan feature engineering, dan daftar cleaning
Lonjakan aktivitas tidak otomatis salah karena dapat berasal dari migrasi atau RECON massal. Dismantle tanpa installation sebelumnya juga dapat berarti histori awal terpotong. Karena itu laporan memberi tindakan yang berbeda untuk setiap masalah dan menyediakan daftar `journey_id` yang dapat direview.

In [ ]:
timeline_anomaly = query("""WITH ordered AS (SELECT o.*, MAX(created_on) FILTER (WHERE status_clean='INSTALLED') OVER (PARTITION BY item_identifier_clean ORDER BY created_on, journey_id ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) prior_installed_on FROM analytics.item_journey_operational_timeline o) SELECT COUNT(*) FILTER (WHERE status_clean='DISMANTLED' AND prior_installed_on IS NULL) dismantle_without_prior_install, COUNT(*) FILTER (WHERE days_since_previous_operational_event<0) negative_gap, COUNT(*) FILTER (WHERE days_since_previous_operational_event=0) zero_gap, COUNT(*) FILTER (WHERE days_since_previous_operational_event>3652.5) gap_gt_10y FROM ordered""")
daily_anomaly_summary = query("SELECT COUNT(*) FILTER (WHERE is_extreme_activity_day) extreme_days, MAX(event_count) max_daily_events, ROUND(AVG(event_count)::numeric,2) average_daily_events, MAX(extreme_event_limit) extreme_limit FROM analytics.eda_daily_activity_anomaly")
display(timeline_anomaly.rename(columns={'dismantle_without_prior_install': 'Dismantle tanpa installation sebelumnya', 'negative_gap': 'Urutan waktu negatif', 'zero_gap': 'Event dengan gap nol', 'gap_gt_10y': 'Gap lebih dari 10 tahun'}))
display(daily_anomaly_summary.rename(columns={'extreme_days': 'Hari dengan lonjakan ekstrem', 'max_daily_events': 'Event terbanyak dalam satu hari', 'average_daily_events': 'Rata-rata event harian', 'extreme_limit': 'Batas outlier harian'}))
top_daily_anomaly = query("SELECT activity_date, event_count, installation_count, dismantle_count, admin_recon_count, bulk_warehouse_reception_count, extreme_event_limit FROM analytics.eda_daily_activity_anomaly WHERE is_extreme_activity_day ORDER BY event_count DESC LIMIT 20")
display(top_daily_anomaly.rename(columns={'activity_date': 'Tanggal', 'event_count': 'Semua event', 'installation_count': 'Installed', 'dismantle_count': 'Dismantled', 'admin_recon_count': 'RECON administratif', 'bulk_warehouse_reception_count': 'Penerimaan gudang massal', 'extreme_event_limit': 'Batas outlier'}))
largest_spike = top_daily_anomaly.iloc[0]
display(Markdown(f"Lonjakan terbesar terjadi pada **{largest_spike.activity_date:%d-%m-%Y}** sebanyak **{int(largest_spike.event_count):,} event**. Rinciannya: **{int(largest_spike.bulk_warehouse_reception_count):,} penerimaan gudang massal**, **{int(largest_spike.installation_count):,} installation**, **{int(largest_spike.dismantle_count):,} dismantle**, dan **{int(largest_spike.admin_recon_count):,} RECON administratif**. Karena itu lonjakan terbesar bukan failure maupun pemasangan.".replace(',', '.')))
daily_activity = query("SELECT activity_date, event_count, is_extreme_activity_day FROM analytics.eda_daily_activity_anomaly ORDER BY activity_date"); daily_activity['activity_date'] = pd.to_datetime(daily_activity.activity_date); daily_activity['event_count'] = pd.to_numeric(daily_activity.event_count)
plt.figure(figsize=(12, 4)); sns.lineplot(data=daily_activity, x='activity_date', y='event_count', color='steelblue', linewidth=.7); extreme_plot=daily_activity[daily_activity.is_extreme_activity_day]; plt.scatter(extreme_plot.activity_date, extreme_plot.event_count, color='crimson', s=12, label='Lonjakan ekstrem'); plt.yscale('symlog'); plt.title('Aktivitas harian dan tanggal dengan lonjakan ekstrem'); plt.xlabel('Tanggal'); plt.ylabel('Jumlah event (skala symlog)'); plt.legend(); plt.tight_layout(); plt.show()
cleaning_actions = query("SELECT suggested_action, COUNT(*) affected_rows FROM analytics.eda_cleaning_review_detail GROUP BY suggested_action ORDER BY affected_rows DESC")
display(cleaning_actions.rename(columns={'suggested_action': 'Tindakan yang disarankan', 'affected_rows': 'Jumlah baris'}))
cleaning_sample = query("SELECT journey_id, item_model_code_clean, status_clean, place_clean, created_on, review_issues::text, suggested_action FROM analytics.eda_cleaning_review_detail ORDER BY suggested_action, journey_id LIMIT 30")
display(cleaning_sample.rename(columns={'journey_id': 'Journey ID', 'item_model_code_clean': 'Model', 'status_clean': 'Status', 'place_clean': 'Lokasi mentah', 'created_on': 'Waktu', 'review_issues': 'Masalah', 'suggested_action': 'Tindakan'}))
cleaning_policy = pd.DataFrame([['RECON administratif','Simpan untuk audit; keluarkan dari perhitungan waktu'],['Penerimaan gudang massal','Simpan sebagai event operasional dengan semantic khusus; jangan artikan sebagai install/dismantle/failure'],['Tanggal invalid/masa depan','Keluarkan dari timeline/model dan periksa sumber'],['Identifier/model inti kosong','Keluarkan dari pembentukan cycle'],['Model tidak konsisten','Keluarkan dari cohort awal model'],['Fuzzy skor tinggi dan margin aman','Gunakan nama canonical; simpan nama sumber, skor, dan metode mapping'],['Fuzzy skor rendah/ambigu','Jangan auto-map; masukkan daftar review'],['Lokasi tidak cocok master','Simpan event; jangan gunakan fitur lokasi'],['Isi log identik','Review journey_id; deduplikasi hanya setelah dikonfirmasi'],['Cycle durasi nol/negatif','Keluarkan dari cohort'],['Snapshot follow-up belum lengkap','Keluarkan dari training'],['Failure tanpa flow lanjutan','Tetap positif jika onset valid; tandai untuk review'],['Cycle masih berjalan','Gunakan hanya snapshot dengan follow-up 30 hari penuh']], columns=['Kondisi','Perlakuan'])
display(cleaning_policy)
calendar_feature_check = query("SELECT COUNT(*) FILTER (WHERE observation_month NOT BETWEEN 1 AND 12) invalid_month, COUNT(*) FILTER (WHERE observation_day_of_week NOT BETWEEN 1 AND 7) invalid_day, COUNT(*) FILTER (WHERE is_weekend IS DISTINCT FROM (observation_day_of_week IN (6,7))) invalid_weekend FROM analytics.item_observation_30d")
display(calendar_feature_check.rename(columns={'invalid_month': 'Bulan invalid', 'invalid_day': 'Hari invalid', 'invalid_weekend': 'Flag weekend tidak sesuai'}))
feature_decisions = pd.DataFrame([['Target sangat imbalance','Gunakan stratified temporal evaluation; nilai precision, recall, PR-AUC, ROC-AUC, calibration; class weight/resampling hanya pada train'],['Model dan tipe PART','Gunakan; kategori utama item'],['Client canonical','Gunakan dengan minimum support; kelompokkan kategori langka bila perlu'],['Lokasi canonical','Coverage aman; uji model dengan dan tanpa lokasi, UNKNOWN, dan missing flag'],['Umur sejak installation','Gunakan bila tidak redundan setelah seleksi korelasi'],['Bulan, kuartal, hari, weekend','Gunakan sebagai kandidat; validasi kestabilan antarperiode'],['Event/corrective/preventive 30-90-180 hari','Pilih window yang tidak redundan atau gunakan regularisasi; keputusan berdasarkan temporal validation'],['Failure 365 hari sebelumnya','Gunakan sebagai kandidat'],['Waktu sejak corrective/failure terakhir yang kosong','Missing bersifat struktural; gunakan indikator belum pernah + sentinel, bukan imputasi nol tanpa flag'],['Lama di lokasi terakhir','Gunakan hanya dari event masa lalu dan lokasi valid; sertakan missing flag'],['Fitur dengan PSI >=0,25','Jangan langsung dibuang; cek perubahan proses/data, retrain policy, dan monitoring drift'],['Interaksi model-lokasi','Uji dengan regularisasi/minimum support; jangan langsung membuat ribuan kategori'],['Status/outcome setelah snapshot','Dilarang karena leakage']], columns=['Fitur/ide','Keputusan EDA'])
display(feature_decisions)

## 20. Kesimpulan EDA
Kesimpulan berikut dibuat otomatis dari hasil pemeriksaan terbaru.

In [ ]:
weekly = cadence.loc[cadence.cadence_days.eq(7)].iloc[0]
monthly = cadence.loc[cadence.cadence_days.eq(30)].iloc[0]
ongoing_count = int(incomplete.loc[incomplete.followup_review_group.eq('LIKELY_ONGOING_0_30D'), 'failure_count'].iloc[0])
history_gap_count = int(incomplete.loc[incomplete.followup_review_group.eq('LIKELY_HISTORY_GAP_GT_180D'), 'failure_count'].iloc[0])
display(Markdown(f"""**Kesimpulan sederhana**

- Snapshot 7 hari dan 30 hari sama-sama melewatkan **{int(weekly.uncaptured_failure_cycles)}** dan **{int(monthly.uncaptured_failure_cycles)}** failure.
- Snapshot mingguan memberi rata-rata **{weekly.average_positive_snapshots_per_failure:.2f}** peringatan untuk satu failure; snapshot 30 hari memberi **{monthly.average_positive_snapshots_per_failure:.2f}**.
- Ada **{ongoing_count}** failure yang kemungkinan masih berjalan karena terjadi maksimal 30 hari sebelum data berakhir.
- Ada **{history_gap_count}** failure lama yang lebih mungkin memiliki histori lanjutan tidak lengkap.
- Ada **{len(missing_onset)}** kandidat status rusak tanpa tanggal awal failure yang dapat dipercaya; kasus ini belum menjadi label utama.
- Target sangat tidak seimbang: positive rate hanya **{float(positive_row.class_percentage):.4f}%**, atau sekitar **1 positif berbanding {float(positive_row.negative_to_positive_ratio):.2f} negatif**.
- Audit korelasi menemukan **{len(redundant_pairs)} pasangan fitur** dengan |Spearman| minimal 0,80; pasangan tersebut perlu seleksi atau regularisasi, bukan dimasukkan seluruhnya tanpa evaluasi.
- Screening IV teratas adalah **{iv_result.iloc[0].feature}** dengan IV **{iv_result.iloc[0].information_value:.4f}**; hasil ini masih univariat dan wajib diuji dengan split waktu.
- Analisis PSI menemukan **{int(psi_result.psi.ge(.25).sum())} kombinasi fitur-tahun** dengan drift besar terhadap 2024; 2026 masih merupakan periode parsial.
- Unmatched lokasi hanya mengenai **{int(location_coverage.unmatched_snapshot)} snapshot ({float(location_coverage.unmatched_percentage):.4f}%)** dari data training; fitur lokasi boleh diuji dengan UNKNOWN/missing flag dan pembanding tanpa lokasi.
- Audit journal menemukan **{int(journal_quality.loc[journal_quality.check_name.eq('EXACT_LOG_DUPLICATE_EXTRA_ROWS'), 'affected_count'].iloc[0])}** baris tambahan dengan isi identik, **{int(journal_quality.loc[journal_quality.check_name.eq('INVALID_OR_FUTURE_DATE'), 'affected_count'].iloc[0])}** tanggal invalid/masa depan, dan **{int(journal_quality.loc[journal_quality.check_name.eq('LOCATION_NOT_IN_MASTER'), 'affected_count'].iloc[0])}** event lokasi non-master.
- Typo client `KERETE COMMUTER INDONESIA (KCI)` dipetakan secara fuzzy ke `KERETA COMMUTER INDONESIA (KCI)` dengan skor dan margin yang lolos batas aman.
- `GUDANG NUTECH` dipetakan ke `GUDANG NI` melalui alias kontekstual; `NOC JUANDA` tetap review karena kandidatnya ambigu.
- Lonjakan **{int(largest_spike.event_count):,} event** terbesar adalah penerimaan gudang massal, bukan installation, dismantle, atau failure.
- Lifecycle yang lokasi installation dan dismantle-nya sama memiliki median **{float(lifecycle_overall.median_days.iloc[0]):,.2f} hari**; cycle tanpa dismantle tidak dipaksa menjadi nol.
- Lokasi hanya digunakan apabila cocok dengan master lokasi.
- Fitur kalender bulan, kuartal, hari dalam minggu, dan weekend sudah ditambahkan tanpa memakai informasi setelah snapshot.
- Untuk baseline training, snapshot 30 hari lebih ringkas: semua failure tetap tertangkap dan satu failure tidak diulang hampir empat kali.
- Saat dipakai nanti, model tetap dapat dijalankan setiap hari atau setiap ada event baru; jadwal scoring tidak harus mengikuti jarak snapshot training.
- Data siap dilanjutkan ke baseline model, sambil mereview sampel kasus histori lama dan membandingkan performa model dengan serta tanpa fitur lokasi."""))